# SmolVLA baseline on LIBERO-10 with `lerobot-eval` (Kaggle, 2 x T4)

Re-runs the protocol of `baseline_performance/results_300926/` through `baseline_performance/evaluate_all.py`:
10 tasks x seeds 0, 1, 2 x 10 episodes (init states 0-9), each (task, seed) in its own `lerobot-eval`
process, several at a time across both GPUs. Step 9 compares the result per task and per init state with
the local run.

**Kaggle settings (right-hand panel), before running**

1. Accelerator: **GPU T4 x2**. Not P100: the pinned PyTorch build does not support it.
2. Internet: **On** (needs a phone-verified Kaggle account). Packages, the repo and the model are downloaded.

**How to run.** Use **Save Version > Save & Run All (Commit)**. The notebook then runs in the background with
the browser closed, for at most 12 hours, and everything in `/kaggle/working/baseline_lerobot_eval/` is kept as
the version's output. The run stops starting new evaluations after 9 hours and stops completely after 11, so
the output is always saved.

**If it does not finish in one session** (step 9 then says INCOMPLETE): open the notebook, **Add Input >
Your Work >** this notebook, select the last version's output, and Save & Run All again. Step 6 copies the
finished evaluations in and only the missing ones run.

Keep every evaluation of this run on Kaggle; do not pool its episodes with local or Colab ones.

## 1. Settings

In [ ]:
import csv, glob, json, os, re, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "main"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

OUT_NAME = "baseline_lerobot_eval"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"  # results, kept as the version's output

TASKS = "0 1 2 3 4 5 6 7 8 9"
SEEDS = "0 1 2"
N_EPISODES = 10                    # init states 0-9, the protocol of results_300926
LAUNCH_DEADLINE_MIN = 9 * 60       # Kaggle stops a session after 12 hours
HARD_STOP_MIN = 11 * 60
GB_PER_JOB = 5                     # one lerobot-eval process held about 4.5 GB of RAM locally

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

A separate virtual environment with the package versions of the local `lerobot_env`. If the GPU driver is
too old for the CUDA 13 build of PyTorch, the same PyTorch version is installed in a CUDA 12 build instead
and the choice is printed. Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks

Checks that PyTorch sees both GPUs and can run bfloat16 on them (SmolVLA runs in bfloat16; the T4 has no
native support, so this can be slow but must work), and that MuJoCo can render headless. If EGL rendering
fails, the cell switches to CPU rendering (OSMesa), which is slower; the backend is recorded in
`provenance.jsonl`.

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1")

sh(f"""{PY} -c "
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16)
print('bfloat16 matmul ok', float((x @ x).float().abs().mean()) > 0)
" """, env=RUN_ENV)

egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
if "EGL context ok" not in egl.stdout:
    print("EGL rendering failed; switching to OSMesa (CPU rendering, slower).")
    sh("apt-get install -y -qq libosmesa6-dev > /dev/null")
    RUN_ENV.update(MUJOCO_GL="osmesa", PYOPENGL_PLATFORM="osmesa")
    sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('OSMesa context ok')\"", env=RUN_ENV)
print("render backend:", RUN_ENV["MUJOCO_GL"])
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)

## 6. Resume from an earlier version's output

If an earlier version's output was added as input, its finished evaluations are copied in. Nothing happens
on a first run.

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
for prev in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True)):
    n = len(glob.glob(f"{prev}/task_*_seed_*/eval_info.json"))
    print(f"copying {n} finished evaluations from {prev}")
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
done = len(glob.glob(f"{OUT_ROOT}/task_*_seed_*/eval_info.json"))
print(f"{done} of {len(TASKS.split()) * len(SEEDS.split())} evaluations already finished")

## 7. Timing check (one episode, not counted)

One episode of task 3 in a separate folder. The base policy solves most task 3 episodes, so a failure here
is worth a look in `_timing/task_3_seed_0/eval.log`, but does not stop the run. Skipped when resuming.

In [ ]:
if done == 0:
    t0 = time.time()
    sh(f"cd {REPO} && {PY} baseline_performance/evaluate_all.py --resume_dir {OUT_ROOT}/_timing --tasks 3 --seeds 0 "
       f"--n_episodes 1 --lerobot_eval {VENV}/bin/lerobot-eval --hard_stop_min 60", check=False, env=RUN_ENV)
    print(open(f"{OUT_ROOT}/_timing/consolidated_summary.txt").read())
    print(f"{(time.time() - t0) / 60:.1f} min for one episode, including the model download and load")
else:
    print("resuming; timing check skipped")

## 8. Run

Runs in the foreground so that Save & Run All waits for it. Prints progress every 10 minutes; the full log is
`evaluate_all.runner.log` in the output folder.

In [ ]:
cmd = (f"cd {REPO} && {PY} -u baseline_performance/evaluate_all.py --resume_dir {OUT_ROOT} --tasks {TASKS} "
       f"--seeds {SEEDS} --n_episodes {N_EPISODES} --lerobot_eval {VENV}/bin/lerobot-eval --jobs {JOBS} "
       f"--gpus {GPU_IDS} --launch_deadline_min {LAUNCH_DEADLINE_MIN} --hard_stop_min {HARD_STOP_MIN}")
print("$", cmd)
log_path = f"{OUT_ROOT}/evaluate_all.runner.log"
with open(log_path, "a") as log:
    proc = subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=RUN_ENV, stdout=log, stderr=subprocess.STDOUT)
    while proc.poll() is None:
        time.sleep(600)
        n = len(glob.glob(f"{OUT_ROOT}/task_*_seed_*/eval_info.json"))
        print(time.strftime("%H:%M"), f"{n} evaluations finished", flush=True)
print("\n".join(open(log_path).read().splitlines()[-15:]))
RUN_EXIT = proc.returncode
print("exit code", RUN_EXIT, "(2 = incomplete; add this version's output as input and run again)")

## 9. Results and comparison with the local run

Per task, Kaggle against `results_300926` (local, same init states and seeds), and per (task, init state)
the number of the three seeds that succeeded on each machine.

In [ ]:
print(open(f"{OUT_ROOT}/consolidated_summary.txt").read())

def load(path):
    with open(path) as f:
        return {(int(r["task_id"]), int(r["seed"]), int(r["init_state"])): int(r["success"]) for r in csv.DictReader(f)}

here = load(f"{OUT_ROOT}/per_episode.csv")
local = load(f"{REPO}/baseline_performance/results_300926/per_episode.csv")
print("task | local successes | Kaggle successes | init states solved by local but never on Kaggle, and the reverse")
for t in sorted({k[0] for k in here}):
    keys = [k for k in local if k[0] == t and k in here]
    if len({k[1] for k in keys}) < len(SEEDS.split()):
        print(f"{t:4d} | incomplete")
        continue
    per_state = lambda d, i: sum(d[k] for k in keys if k[2] == i)
    states = sorted({k[2] for k in keys})
    only_local = [i for i in states if per_state(local, i) and not per_state(here, i)]
    only_here = [i for i in states if per_state(here, i) and not per_state(local, i)]
    print(f"{t:4d} | {sum(local[k] for k in keys):3d}/{len(keys)} | {sum(here[k] for k in keys):3d}/{len(keys)} | {only_local} | {only_here}")

## 10. Files for the repo

The output folder (with videos) is in the version's **Output** tab. This zip holds only the small files; unpack
it into `week6/baseline_lerobot_eval_kaggle/` in the repo.

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/{OUT_NAME}_results.zip . -x '*/videos/*' '*.mp4'")
sh(f"du -h /kaggle/working/{OUT_NAME}_results.zip")